# 🏠 SmartThings 기반 홈 IoT 멀티 디바이스 제어 에이전트

**GGUF 4-bit 양자화 모델 + LangChain v1.0 Sub-Agent 패턴**

## 📌 프로젝트 개요

SmartThings 기반 홈 IoT 환경에서, 센서 데이터와 사용자 상황을 분석하여
상황에 맞는 디바이스 제어 + 브리핑을 자동으로 제공하는 멀티 에이전트 시스템입니다.

## 🏗️ 아키텍처

```
┌──────────────────────────────────────┐
│       Supervisor Agent               │
│    (홈 IoT 작업 관리자)                │
└──────────┬───────────────────────────┘
           │
     ┌─────┼─────────┬──────────┐
     ▼     ▼         ▼          ▼
┌────────┐┌────────┐┌─────────┐┌─────────┐
│Context ││Device  ││Briefing ││Schedule │
│Analyzer││Control ││ Agent   ││ Agent   │
│(상황분석)││(디바이스)││(브리핑)   ││(일정)    │
└────────┘└────────┘└─────────┘└─────────┘
```

## 🔑 핵심 패턴

- **11번 참조**: GGUF 모델을 llama-server로 로드 → `ChatOpenAI(base_url=...)` 연결
- **프로젝트2 참조**: 서브에이전트를 `@tool`로 래핑하여 Supervisor가 호출

## 📂 더미데이터 시나리오

| 시나리오 | 시간 | 센서 | 상황 | 에이전트 동작 |
|---------|------|------|------|-------------|
| 아침 출근 | 09:00 | 운동센서 | 출근 중 | 하루 브리핑 + 외출 모드 |
| 저녁 퇴근 | 20:00 | 위치센서 | 퇴근 중 | 일정 안내 + 귀가 모드 |
| 취침 | 23:00 | 수면센서 | 수면 준비 | 취침 모드 전환 |
| 주말 아침 | 10:00 | 운동센서 | 재택 | 릴랙스 모드 |
| 외출 감지 | 14:00 | 도어센서 | 외출 | 에너지 절약 모드 |

### 참조 문서
- [LangChain v1.0 Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [create_agent API Reference](https://reference.langchain.com/python/langchain/agents/factory/create_agent)
- [SmartThings Devices API](https://developer.smartthings.com/docs/devices/device-basics)


## 1. 패키지 설치
* 필요시 패키지 설치를 진행합니다.


In [ ]:
# !pip install -U "langchain>=1.0" langchain-openai

# print("✅ 설치 완료!")
# print("⚠️ 버전 적용을 위해 런타임을 재시작해야 합니다.")


## 2. 설정값

> ⚠️ `Q4_GGUF_PATH`를 실제 GGUF 모델 파일 경로로 변경하세요.


In [ ]:
import os

# ─── GGUF 모델 경로 ───
GGUF_MODEL_DIR = "/home/user/MultiAgents/Day_2/Ex/qwen3-vision-finetune-edu/gguf_models"

Q4_GGUF_PATH = os.path.join(GGUF_MODEL_DIR, "qwen3-vl-q4_k_m.gguf")
MMPROJ_GGUF_PATH = os.path.join(GGUF_MODEL_DIR, "qwen3-vl-mmproj-f16.gguf")

# ─── 서버 설정 ───
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8181
LLAMA_SERVER_BIN = "/home/user/MultiAgents/Day_2/Ex/llama.cpp/build/bin/llama-server"

# ─── 비전(멀티모달) 사용 여부 ───
USE_VISION = True  # False로 변경하면 텍스트 전용 모드

# ─── 확인 ───
print("📁 설정 확인:")
print(f"  GGUF 모델:     {Q4_GGUF_PATH}  {'✅' if os.path.exists(Q4_GGUF_PATH) else '❌ 파일 없음'}")
print(f"  비전 모델:     {MMPROJ_GGUF_PATH}  {'✅' if os.path.exists(MMPROJ_GGUF_PATH) else '⚠️ 파일 없음 (텍스트 전용)'}")
print(f"  llama-server:  {LLAMA_SERVER_BIN}  {'✅' if os.path.exists(LLAMA_SERVER_BIN) else '❌ 바이너리 없음'}")
print(f"  서버 주소:     http://{SERVER_HOST}:{SERVER_PORT}")
print(f"  비전 사용:     {USE_VISION}")

## 3. llama-server 실행

llama-server를 OpenAI 호환 API 서버로 실행합니다.

> 핵심 옵션:
> - `--jinja` : tool calling 지원
> - `-ngl 99` : GPU 전체 오프로드
> - `-c 4096` : 컨텍스트 길이


In [ ]:
import subprocess
import time
import requests

def start_llama_server():
    """llama-server를 OpenAI 호환 API 서버로 실행합니다."""

    # 모델 파일 존재 여부 확인
    if not os.path.exists(Q4_GGUF_PATH):
        raise FileNotFoundError(
            f"❌ GGUF 모델 파일을 찾을 수 없습니다: {Q4_GGUF_PATH}\n"
            f"   Q4_GGUF_PATH를 실제 모델 경로로 수정하세요."
        )

    if not os.path.exists(LLAMA_SERVER_BIN):
        raise FileNotFoundError(
            f"❌ llama-server를 찾을 수 없습니다: {LLAMA_SERVER_BIN}\n"
            f"   llama.cpp 빌드 셀을 먼저 실행하세요."
        )

    # 이전 프로세스 정리
    os.system("pkill -f llama-server 2>/dev/null || true")
    time.sleep(2)

    # 서버 시작 명령 구성
    cmd = [
        LLAMA_SERVER_BIN,
        "-m", Q4_GGUF_PATH,
        "--host", SERVER_HOST,
        "--port", str(SERVER_PORT),
        "--jinja",              # ← tool calling 지원의 핵심!
        "-fa", "auto",          # Flash Attention
        "-ngl", "99",           # GPU 전체 오프로드
        "-c", "4096",           # 컨텍스트 길이
        "--temp", "0.1",
    ]

    # 비전 모델 사용 시 mmproj 추가
    if USE_VISION and os.path.exists(MMPROJ_GGUF_PATH):
        cmd.extend(["--mmproj", MMPROJ_GGUF_PATH])
        print(f"🖼️  비전 모델 활성화: {os.path.basename(MMPROJ_GGUF_PATH)}")

    print(f"🚀 llama-server를 시작합니다...")
    print(f"   텍스트 모델: {os.path.basename(Q4_GGUF_PATH)}")

    server_process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    # 헬스체크 (최대 120초 대기)
    print("⏳ 서버 준비 대기 중 (모델 로드에 시간이 걸릴 수 있습니다)...")
    for i in range(120):
        try:
            resp = requests.get(
                f"http://{SERVER_HOST}:{SERVER_PORT}/health", timeout=2
            )
            if resp.status_code == 200:
                print(f"✅ llama-server 준비 완료! (http://{SERVER_HOST}:{SERVER_PORT})")
                return server_process
        except requests.ConnectionError:
            pass
        time.sleep(1)

    # 실패 시 로그 출력
    stderr_output = server_process.stderr.read().decode()
    server_process.terminate()
    raise RuntimeError(f"❌ 서버 시작 실패.\n로그:\n{stderr_output}")


# 서버 시작
server_process = start_llama_server()

# 서버 속성 확인
try:
    props = requests.get(f"http://{SERVER_HOST}:{SERVER_PORT}/props").json()
    print(f"📋 Chat template: {props.get('chat_template', 'N/A')[:80]}...")
except Exception as e:
    print(f"⚠️ 속성 확인 실패: {e}")

## 4. 더미데이터 정의 — SmartThings 홈 IoT 환경

SmartThings API의 **Capability** 기반 디바이스 모델링을 참고하여 더미데이터를 구성합니다.

### 구성 요소
- 👤 **사용자 프로필**: 이름, 주소, 생활 패턴, 선호 설정
- 🏠 **디바이스 목록**: 조명, 에어컨, TV, 스피커, 블라인드, 도어락, 로봇청소기, 공기청정기 (총 13개)
- 📡 **센서 시나리오**: 운동센서, 위치센서, 온습도센서, 도어센서, 수면센서, 대기질센서
- 📅 **일정 데이터**: 캘린더 이벤트 5개
- 🌤️ **날씨 데이터**: 현재 날씨 + 시간대별 예보 + 기상특보



In [ ]:
import json
from datetime import datetime, timedelta
from typing import Optional

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1) 사용자 프로필
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
USER_PROFILE = {
    "name": "김민준",
    "home_address": "서울시 강남구 삼성로 100",
    "work_address": "서울시 서초구 테헤란로 200",
    "wake_up_time": "07:00",
    "sleep_time": "23:30",
    "work_start": "09:30",
    "work_end": "18:30",
    "preferences": {
        "morning_temp": 24,
        "evening_temp": 22,
        "sleep_temp": 20,
        "preferred_light_color": "warm_white",
        "morning_music": "jazz",
        "evening_music": "lo-fi",
    }
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2) SmartThings 디바이스 목록 (더미)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 각 디바이스의 이름, 위치, 타입, 지원 기능, 현재 상태를 담는 딕셔너리를 정의합니다.
DEVICES = {
    # 거실에 있는 디바이스들을 정의합니다.
    "living_room_light": {
        # 거실 조명의 기본 정보와 상태를 저장합니다.
        "name": "거실 조명", "room": "거실", "type": "light",
        # 조명 on/off, 밝기, 색온도 제어 기능을 지원합니다.
        "capabilities": ["switch", "switchLevel", "colorTemperature"],
        # 현재 전원은 꺼져 있고 밝기와 색온도 상태를 저장합니다.
        "state": {"switch": "off", "level": 0, "colorTemp": 4000}
    },
    "living_room_ac": {
        # 거실 에어컨의 기본 정보와 상태를 저장합니다.
        "name": "거실 에어컨", "room": "거실", "type": "airConditioner",
        # 전원, 냉방 설정 온도, 운전 모드 제어 기능을 지원합니다.
        "capabilities": ["switch", "thermostatCoolingSetpoint", "airConditionerMode"],
        # 현재 전원은 꺼져 있고 설정 온도와 모드를 저장합니다.
        "state": {"switch": "off", "setpoint": 24, "mode": "auto"}
    },
    "living_room_tv": {
        # 거실 TV의 기본 정보와 상태를 저장합니다.
        "name": "거실 TV", "room": "거실", "type": "television",
        # 전원, 재생 제어, 채널 변경 기능을 지원합니다.
        "capabilities": ["switch", "mediaPlayback", "tvChannel"],
        # 현재 전원은 꺼져 있고 채널과 볼륨 상태를 저장합니다.
        "state": {"switch": "off", "channel": "KBS1", "volume": 15}
    },
    "living_room_speaker": {
        # 거실 스피커의 기본 정보와 상태를 저장합니다.
        "name": "거실 스피커", "room": "거실", "type": "speaker",
        # 전원, 미디어 재생, 볼륨 제어 기능을 지원합니다.
        "capabilities": ["switch", "mediaPlayback", "audioVolume"],
        # 현재 전원은 꺼져 있고 재생 중인 항목과 볼륨 상태를 저장합니다.
        "state": {"switch": "off", "playing": None, "volume": 30}
    },
    "living_room_blinds": {
        # 거실 블라인드의 기본 정보와 상태를 저장합니다.
        "name": "거실 블라인드", "room": "거실", "type": "windowShade",
        # 블라인드 열림/닫힘과 높이 조절 기능을 지원합니다.
        "capabilities": ["windowShade", "windowShadeLevel"],
        # 현재 블라인드는 닫혀 있고 높이는 0으로 설정되어 있습니다.
        "state": {"shade": "closed", "level": 0}
    },

    # 침실에 있는 디바이스들을 정의합니다.
    "bedroom_light": {
        # 침실 조명의 기본 정보와 상태를 저장합니다.
        "name": "침실 조명", "room": "침실", "type": "light",
        # 조명 on/off, 밝기, 색온도 제어 기능을 지원합니다.
        "capabilities": ["switch", "switchLevel", "colorTemperature"],
        # 현재 전원은 꺼져 있고 밝기와 색온도 상태를 저장합니다.
        "state": {"switch": "off", "level": 0, "colorTemp": 3000}
    },
    "bedroom_ac": {
        # 침실 에어컨의 기본 정보와 상태를 저장합니다.
        "name": "침실 에어컨", "room": "침실", "type": "airConditioner",
        # 전원과 냉방 설정 온도 제어 기능을 지원합니다.
        "capabilities": ["switch", "thermostatCoolingSetpoint"],
        # 현재 전원은 꺼져 있고 설정 온도와 모드를 저장합니다.
        "state": {"switch": "off", "setpoint": 22, "mode": "sleep"}
    },

    # 주방에 있는 디바이스를 정의합니다.
    "kitchen_light": {
        # 주방 조명의 기본 정보와 상태를 저장합니다.
        "name": "주방 조명", "room": "주방", "type": "light",
        # 조명 on/off와 밝기 제어 기능을 지원합니다.
        "capabilities": ["switch", "switchLevel"],
        # 현재 전원은 꺼져 있고 밝기는 0입니다.
        "state": {"switch": "off", "level": 0}
    },

    # 현관에 있는 디바이스들을 정의합니다.
    "front_door_lock": {
        # 현관 도어락의 기본 정보와 상태를 저장합니다.
        "name": "현관 도어락", "room": "현관", "type": "lock",
        # 잠금 제어 기능을 지원합니다.
        "capabilities": ["lock"],
        # 현재 문은 잠긴 상태입니다.
        "state": {"lock": "locked"}
    },
    "front_door_sensor": {
        # 현관 도어 센서의 기본 정보와 상태를 저장합니다.
        "name": "현관 도어 센서", "room": "현관", "type": "contactSensor",
        # 문 열림/닫힘 감지 기능을 지원합니다.
        "capabilities": ["contactSensor"],
        # 현재 문은 닫힌 상태입니다.
        "state": {"contact": "closed"}
    },

    # 공용으로 사용하는 디바이스들을 정의합니다.
    "robot_vacuum": {
        # 로봇 청소기의 기본 정보와 상태를 저장합니다.
        "name": "로봇 청소기", "room": "거실", "type": "robotVacuum",
        # 전원과 청소기 이동 상태 제어 기능을 지원합니다.
        "capabilities": ["switch", "robotCleanerMovement"],
        # 현재 전원은 꺼져 있고 움직임 상태는 idle입니다.
        "state": {"switch": "off", "movement": "idle"}
    },
    "air_purifier": {
        # 공기청정기의 기본 정보와 상태를 저장합니다.
        "name": "공기청정기", "room": "거실", "type": "airPurifier",
        # 전원과 공기질 측정 기능을 지원합니다.
        "capabilities": ["switch", "airQualitySensor"],
        # 현재 전원은 꺼져 있고 공기질 및 PM2.5 상태를 저장합니다.
        "state": {"switch": "off", "airQuality": 35, "pm25": 12}
    },
}

# 등록된 전체 디바이스 개수를 출력합니다.
print(f"✅ 디바이스 {len(DEVICES)}개 등록 완료!")
# 각 디바이스의 이름, 방 위치, 타입 정보를 한 줄씩 출력합니다.
for did, d in DEVICES.items():
    # 디바이스 이름, 위치, 타입을 보기 좋은 형식으로 출력합니다.
    print(f"   🏷️ {d['name']:10s} [{d['room']:4s}] - {d['type']}")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3) 센서 데이터 (시나리오별 더미)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 상황별 센서 값과 설명을 담는 딕셔너리를 정의합니다.
SENSOR_SCENARIOS = {
    "morning_commute": {
        # 아침 출근 상황의 기준 시각을 저장합니다.
        "timestamp": "2025-02-27T09:00:00",
        # 출근 상황에서의 각종 센서 상태를 저장합니다.
        "sensors": {
            # 현관 모션 센서가 최근 감지된 상태를 기록합니다.
            "motion_sensor": {"room": "현관", "detected": True, "last_activity": "08:55"},
            # 사용자의 현재 위치와 집으로부터 거리를 기록합니다.
            "location_sensor": {"user_location": "출근 중", "distance_from_home_km": 0.5},
            # 현관문 센서의 닫힘 상태와 마지막 열린 시각을 기록합니다.
            "door_sensor": {"contact": "closed", "last_opened": "08:58"},
            # 실내외 온도 정보를 기록합니다.
            "temperature_sensor": {"indoor": 23.5, "outdoor": -2.0},
            # 실내외 습도 정보를 기록합니다.
            "humidity_sensor": {"indoor": 45, "outdoor": 30},
            # 미세먼지와 대기질 상태를 기록합니다.
            "air_quality_sensor": {"pm25": 35, "pm10": 52, "grade": "보통"},
        },
        # 이 시나리오를 설명하는 요약 문구입니다.
        "description": "아침 9시, 운동 센서 감지, 현관문 닫힘 → 출근 중"
    },
    "evening_return": {
        # 저녁 퇴근 상황의 기준 시각을 저장합니다.
        "timestamp": "2025-02-27T20:00:00",
        # 퇴근 상황에서의 각종 센서 상태를 저장합니다.
        "sensors": {
            # 현재 움직임이 감지되지 않는 상태를 기록합니다.
            "motion_sensor": {"room": "없음", "detected": False},
            # 사용자가 퇴근 중이며 집까지 남은 거리를 기록합니다.
            "location_sensor": {"user_location": "퇴근 중", "distance_from_home_km": 3.2},
            # 현관문이 닫혀 있는 상태를 기록합니다.
            "door_sensor": {"contact": "closed"},
            # 실내외 온도 정보를 기록합니다.
            "temperature_sensor": {"indoor": 21.0, "outdoor": 1.5},
            # 실내외 습도 정보를 기록합니다.
            "humidity_sensor": {"indoor": 40, "outdoor": 55},
            # 미세먼지와 대기질 상태를 기록합니다.
            "air_quality_sensor": {"pm25": 28, "pm10": 41, "grade": "좋음"},
        },
        # 이 시나리오를 설명하는 요약 문구입니다.
        "description": "저녁 8시, 위치 센서 감지, 퇴근 중 → 집 근처 도착 예정"
    },
    "bedtime": {
        # 취침 전 상황의 기준 시각을 저장합니다.
        "timestamp": "2025-02-27T23:00:00",
        # 취침 상황에서의 각종 센서 상태를 저장합니다.
        "sensors": {
            # 침실에서 움직임이 감지된 상태를 기록합니다.
            "motion_sensor": {"room": "침실", "detected": True, "last_activity": "22:50"},
            # 사용자가 집 안에 있는 상태를 기록합니다.
            "location_sensor": {"user_location": "집", "distance_from_home_km": 0},
            # 수면 준비 상태와 생체 신호를 기록합니다.
            "sleep_sensor": {"status": "수면 준비", "heart_rate": 62, "activity_level": "low"},
            # 실내외 온도 정보를 기록합니다.
            "temperature_sensor": {"indoor": 22.0, "outdoor": -1.0},
            # 주변 조도 값을 기록합니다.
            "light_sensor": {"ambient_lux": 15},
        },
        # 이 시나리오를 설명하는 요약 문구입니다.
        "description": "밤 11시, 수면 센서 감지, 침실 활동 → 취침 모드"
    },
    "weekend_morning": {
        # 주말 아침 상황의 기준 시각을 저장합니다.
        "timestamp": "2025-03-01T10:00:00",
        # 주말 오전 상황에서의 각종 센서 상태를 저장합니다.
        "sensors": {
            # 거실에서 최근 움직임이 감지된 상태를 기록합니다.
            "motion_sensor": {"room": "거실", "detected": True, "last_activity": "09:45"},
            # 사용자가 집 안에 있는 상태를 기록합니다.
            "location_sensor": {"user_location": "집", "distance_from_home_km": 0},
            # 실내외 온도 정보를 기록합니다.
            "temperature_sensor": {"indoor": 22.5, "outdoor": 5.0},
            # 실내외 습도 정보를 기록합니다.
            "humidity_sensor": {"indoor": 42, "outdoor": 40},
            # 미세먼지와 대기질 상태를 기록합니다.
            "air_quality_sensor": {"pm25": 15, "pm10": 22, "grade": "좋음"},
        },
        # 이 시나리오를 설명하는 요약 문구입니다.
        "description": "주말 오전 10시, 거실 활동 감지 → 릴랙스 모드"
    },
    "away_detected": {
        # 외출 감지 상황의 기준 시각을 저장합니다.
        "timestamp": "2025-02-27T14:00:00",
        # 외출 중 상황에서의 각종 센서 상태를 저장합니다.
        "sensors": {
            # 일정 시간 동안 활동이 감지되지 않은 상태를 기록합니다.
            "motion_sensor": {"room": "없음", "detected": False, "no_activity_minutes": 30},
            # 사용자가 외출 중이며 집에서 떨어진 거리를 기록합니다.
            "location_sensor": {"user_location": "외출", "distance_from_home_km": 5.0},
            # 현관문이 닫혀 있고 마지막 열린 시각을 기록합니다.
            "door_sensor": {"contact": "closed", "last_opened": "13:25"},
            # 실내외 온도 정보를 기록합니다.
            "temperature_sensor": {"indoor": 24.0, "outdoor": 3.0},
        },
        # 이 시나리오를 설명하는 요약 문구입니다.
        "description": "오후 2시, 30분간 활동 없음 + 외출 감지 → 에너지 절약 모드"
    }
}

# 등록된 센서 시나리오 개수를 출력합니다.
print(f"✅ 센서 시나리오 {len(SENSOR_SCENARIOS)}개 등록 완료!")
# 각 시나리오 키와 설명을 한 줄씩 출력합니다.
for key, sc in SENSOR_SCENARIOS.items():
    # 시나리오 이름과 설명을 보기 좋은 형식으로 출력합니다.
    print(f"   📡 {key:20s} → {sc['description']}")

In [ ]:
# 캘린더 일정 더미 데이터와 날씨 더미 데이터를 정의하는 구간입니다.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4) 일정 데이터 (캘린더 더미)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 일정 정보를 리스트 형태로 저장합니다.
CALENDAR_EVENTS = [
    {
        # 첫 번째 일정의 기본 정보를 저장합니다.
        "id": "evt_001", "title": "팀 스탠드업 미팅",
        # 일정의 시작 시간과 종료 시간을 ISO 형식 문자열로 저장합니다.
        "start": "2025-02-27T09:30:00", "end": "2025-02-27T10:00:00",
        # 일정 장소, 유형, 우선순위를 저장합니다.
        "location": "Zoom", "type": "meeting", "priority": "high"
    },
    {
        # 두 번째 일정의 기본 정보를 저장합니다.
        "id": "evt_002", "title": "프로젝트 리뷰",
        # 일정의 시작 시간과 종료 시간을 ISO 형식 문자열로 저장합니다.
        "start": "2025-02-27T14:00:00", "end": "2025-02-27T15:30:00",
        # 일정 장소, 유형, 우선순위를 저장합니다.
        "location": "회의실 A", "type": "meeting", "priority": "high"
    },
    {
        # 세 번째 일정의 기본 정보를 저장합니다.
        "id": "evt_003", "title": "헬스장 PT",
        # 일정의 시작 시간과 종료 시간을 ISO 형식 문자열로 저장합니다.
        "start": "2025-02-27T19:00:00", "end": "2025-02-27T20:00:00",
        # 일정 장소, 유형, 우선순위를 저장합니다.
        "location": "강남 피트니스", "type": "personal", "priority": "medium"
    },
    {
        # 네 번째 일정의 기본 정보를 저장합니다.
        "id": "evt_004", "title": "아내 생일 준비",
        # 일정의 시작 시간과 종료 시간을 ISO 형식 문자열로 저장합니다.
        "start": "2025-02-27T20:30:00", "end": "2025-02-27T21:30:00",
        # 일정 장소, 유형, 우선순위를 저장합니다.
        "location": "집", "type": "personal", "priority": "high",
        # 일정과 관련된 추가 메모를 저장합니다.
        "memo": "케이크 주문 확인, 꽃 픽업"
    },
    {
        # 다섯 번째 일정의 기본 정보를 저장합니다.
        "id": "evt_005", "title": "주간 회고 작성",
        # 일정의 시작 시간과 종료 시간을 ISO 형식 문자열로 저장합니다.
        "start": "2025-02-27T17:00:00", "end": "2025-02-27T17:30:00",
        # 일정 장소, 유형, 우선순위를 저장합니다.
        "location": "사무실", "type": "task", "priority": "low"
    },
]

# 날씨 더미 데이터를 정의하는 구간입니다.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5) 날씨 데이터 (더미)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 현재 날씨, 예보, 특보 정보를 담는 딕셔너리를 정의합니다.
WEATHER_DATA = {
    "current": {
        # 현재 기온, 체감온도, 습도를 저장합니다.
        "temp": -2, "feels_like": -7, "humidity": 30,
        # 현재 날씨 상태, 풍속, 자외선 지수를 저장합니다.
        "condition": "맑음", "wind_speed": 12, "uv_index": 3,
        # 현재 초미세먼지 수치와 등급을 저장합니다.
        "pm25": 35, "pm25_grade": "보통",
    },
    "forecast": [
        # 오전 예보 정보를 저장합니다.
        {"time": "오전", "temp_high": 3, "temp_low": -3, "condition": "맑음", "precipitation": 0},
        # 오후 예보 정보를 저장합니다.
        {"time": "오후", "temp_high": 5, "temp_low": 0, "condition": "구름 조금", "precipitation": 10},
        # 저녁 예보 정보를 저장합니다.
        {"time": "저녁", "temp_high": 2, "temp_low": -2, "condition": "맑음", "precipitation": 0},
    ],
    # 현재 발효 중인 기상 특보 목록을 저장합니다.
    "alerts": ["한파 주의보: 내일 아침 최저 -10°C 예상"]
}

# 로드된 일정 개수와 날씨 데이터 준비 완료 메시지를 출력합니다.
print(f"✅ 일정 {len(CALENDAR_EVENTS)}개, 날씨 데이터 로드 완료!")
# 사용자 프로필에 저장된 이름을 출력합니다.
print(f"   👤 사용자: {USER_PROFILE['name']}")
# 현재 날씨 상태와 현재 기온을 출력합니다.
print(f"   🌡️ 현재 날씨: {WEATHER_DATA['current']['condition']} {WEATHER_DATA['current']['temp']}°C")

## 5. Tool 정의 — SmartThings IoT 제어 도구

LangChain v1.0의 `@tool` 데코레이터를 사용하여 IoT 제어 도구를 정의합니다.

### 등록 도구 목록

| 도구명 | 설명 |
|-------|------|
| `get_sensor_data` | 센서 데이터 조회 |
| `control_device` | 개별 디바이스 제어 |
| `execute_scene` | 씬(Scene) 일괄 실행 |
| `get_schedule` | 일정 조회 |
| `get_weather` | 날씨 조회 |
| `get_device_status` | 전체 디바이스 상태 조회 |
| `get_user_profile` | 사용자 프로필 조회 |

> 참조: [LangChain v1.0 Tools](https://docs.langchain.com/oss/python/langchain/tools)


In [ ]:
from typing import Optional
from langchain.tools import tool
from pydantic import BaseModel, Field


# ─── 입력 스키마 ───
# 아래 클래스들은 Tool에 전달할 입력 형식을 정의합니다.
# 각 입력 스키마에 필요한 설명을 완성합니다.


# 센서 조회용 입력 스키마입니다.
# 사용자는 하나의 시나리오 키를 전달하고, 그에 맞는 센서 데이터를 조회하게 됩니다.
# 사용 가능한 예시는 다음과 같습니다:
# morning_commute, evening_return, bedtime, weekend_morning, away_detected
class SensorQueryInput(BaseModel):
    # 조회할 센서 시나리오 이름을 문자열로 받습니다.
    # 예: "morning_commute"
    scenario_key: str = Field(
        description="조회할 센서 시나리오 키. 사용 가능: morning_commute, evening_return, bedtime, weekend_morning, away_detected"
    )


# 디바이스 제어용 입력 스키마입니다.
# 어떤 디바이스를 제어할지(device_id), 어떤 명령을 실행할지(command),
# 그리고 필요한 경우 명령에 사용할 추가 값(value)을 받습니다.
# 예를 들어 밝기 조절, 온도 설정, 볼륨 변경 같은 명령은 value가 필요할 수 있습니다.
class DeviceControlInput(BaseModel):
    # 제어 대상이 되는 디바이스의 고유 ID입니다.
    # 예: "living_room_light", "bedroom_ac", "front_door_lock"
    device_id: str = Field(description="제어할 디바이스 ID")

    # 디바이스에 내릴 명령입니다.
    # 예: on, off, setLevel, setTemp, lock, unlock, open, close, setVolume, play
    # 어떤 명령이 가능한지는 디바이스 종류에 따라 달라질 수 있습니다.
    command: str = Field(description="실행할 디바이스 명령. 사용 가능: on, off, setLevel, setTemp, lock, unlock, open, close, setVolume, play")

    # 명령에 따라 추가로 필요한 값입니다.
    # 예:
    # - 온도 설정: "22"
    # - 밝기 설정: "80"
    # - 볼륨 설정: "30"
    # - 재생할 콘텐츠 이름: "jazz playlist"
    # on/off 같은 단순 명령은 None이어도 됩니다.
    value: Optional[str] = Field(
        default=None,
        description="명령 값 (예: 온도 '22', 밝기 '80', 볼륨 '30')"
    )


# 여러 디바이스를 한 번에 제어하는 Scene 실행용 입력 스키마입니다.
# Scene은 미리 정의된 자동화 동작 묶음이라고 생각하면 됩니다.
# 예를 들어 'bedtime' 씬은 조명을 끄고, 블라인드를 닫고, 침실 환경을 수면 모드로 바꿀 수 있습니다.
class DeviceSceneInput(BaseModel):
    # 실행할 씬 이름을 문자열로 받습니다.
    # 사용 가능한 예시는 다음과 같습니다:
    # morning_briefing, go_out, welcome_home, bedtime, relax, energy_saving
    scene_name: str = Field(
        description="실행할 SmartThings 씬 이름. 사용 가능: morning_briefing, go_out, welcome_home, bedtime, relax, energy_saving"
    )


# 일정 조회용 입력 스키마입니다.
# 특정 시간대의 일정만 조회하거나, 다음 일정을 조회할 때 사용합니다.
class ScheduleQueryInput(BaseModel):
    # 조회할 시간 범위를 문자열로 받습니다.
    # 예:
    # - "morning"   : 오전 일정
    # - "afternoon" : 오후 일정
    # - "evening"   : 저녁 일정
    # - "all_day"   : 전체 일정
    # - "next"      : 다음 예정된 일정 1개
    time_range: str = Field(
        description="조회할 일정 시간 범위. 사용 가능: morning, afternoon, evening, all_day, next"
    )


# 날씨 조회용 입력 스키마입니다.
# 현재 날씨, 예보, 기상 특보 중 어떤 정보를 가져올지 지정합니다.
class WeatherQueryInput(BaseModel):
    # 조회할 날씨 정보 유형입니다.
    # 예:
    # - "current"  : 현재 날씨
    # - "forecast" : 날씨 예보
    # - "alerts"   : 기상 특보
    query_type: str = Field(
        description="조회할 날씨 정보 유형. 사용 가능: current, forecast, alerts"
    )

In [ ]:
# 센서, 날씨, 일정, 프로필 조회용 Tool들을 구현하는 구간입니다.
# ─── Tool 구현: 센서, 날씨, 일정, 프로필 조회 ───

# 특정 센서 시나리오의 데이터를 조회하는 Tool을 정의합니다.
@tool("get_sensor_data", args_schema=SensorQueryInput)
def get_sensor_data(scenario_key: str) -> str:
    """SmartThings 센서 데이터를 조회합니다.
    운동센서, 위치센서, 온습도센서, 도어센서, 대기질센서 등의 현재 상태를 반환합니다."""

    # 요청한 시나리오 키가 등록된 센서 시나리오에 있는지 확인합니다.
    if scenario_key in SENSOR_SCENARIOS:
        # 해당 시나리오 데이터를 가져옵니다.
        scenario = SENSOR_SCENARIOS[scenario_key]
        # 시나리오 데이터를 보기 좋은 JSON 문자열로 반환합니다.
        return json.dumps(scenario, ensure_ascii=False, indent=2)
    # 사용 가능한 시나리오 키 목록을 문자열로 만듭니다.
    available = ", ".join(SENSOR_SCENARIOS.keys())
    # 알 수 없는 시나리오일 경우 사용 가능한 목록과 함께 안내 메시지를 반환합니다.
    return f"알 수 없는 시나리오: {scenario_key}. 사용 가능: {available}"


# 사용자의 일정 데이터를 시간대 기준으로 조회하는 Tool을 정의합니다.
@tool("get_schedule", args_schema=ScheduleQueryInput)
def get_schedule(time_range: str) -> str:
    """오늘의 일정을 시간대별로 조회합니다.
    morning(오전), afternoon(오후), evening(저녁), all_day(전체), next(다음 일정)를 지원합니다."""

    # 시간대별 조회 범위를 시 단위 튜플로 정의합니다.
    time_filters = {
        "morning": (6, 12), "afternoon": (12, 18),
        "evening": (18, 24), "all_day": (0, 24),
    }
    # 다음 일정만 조회하는 경우를 처리합니다.
    if time_range == "next":
        # 현재 시각을 가져옵니다.
        now = datetime.now()
        # 현재 시각 이후에 시작하는 일정만 추립니다.
        upcoming = [e for e in CALENDAR_EVENTS if datetime.fromisoformat(e["start"]) > now]
        # 예정된 일정이 하나 이상 있으면 가장 빠른 일정을 반환합니다.
        if upcoming:
            # 시작 시각 기준으로 오름차순 정렬합니다.
            upcoming.sort(key=lambda x: x["start"])
            # 가장 가까운 다음 일정을 JSON 문자열로 반환합니다.
            return json.dumps(upcoming[0], ensure_ascii=False, indent=2)
        # 예정된 다음 일정이 없으면 안내 메시지를 반환합니다.
        return "다음 일정이 없습니다."
    # morning, afternoon, evening, all_day 중 하나인지 확인합니다.
    if time_range in time_filters:
        # 선택한 시간대의 시작 시각과 종료 시각을 가져옵니다.
        start_h, end_h = time_filters[time_range]
        # 해당 시간대에 시작하는 일정만 필터링합니다.
        filtered = [e for e in CALENDAR_EVENTS if start_h <= datetime.fromisoformat(e["start"]).hour < end_h]
        # 필터링된 일정이 있으면 JSON 문자열로 반환합니다.
        if filtered:
            return json.dumps(filtered, ensure_ascii=False, indent=2)
        # 해당 시간대 일정이 없으면 안내 메시지를 반환합니다.
        return f"{time_range} 시간대에 일정이 없습니다."
    # 지원하지 않는 시간대 요청일 경우 사용 가능한 값을 안내합니다.
    return f"알 수 없는 시간대: {time_range}. 사용 가능: morning, afternoon, evening, all_day, next"


# 날씨 정보를 유형별로 조회하는 Tool을 정의합니다.
@tool("get_weather", args_schema=WeatherQueryInput)
def get_weather(query_type: str) -> str:
    """현재 날씨, 예보, 기상 특보를 조회합니다."""

    # 현재 날씨 조회 요청이면 current 데이터를 반환합니다.
    if query_type == "current":
        return json.dumps(WEATHER_DATA["current"], ensure_ascii=False, indent=2)
    # 일기예보 조회 요청이면 forecast 데이터를 반환합니다.
    elif query_type == "forecast":
        return json.dumps(WEATHER_DATA["forecast"], ensure_ascii=False, indent=2)
    # 기상특보 조회 요청이면 alerts 데이터를 반환합니다.
    elif query_type == "alerts":
        return json.dumps(WEATHER_DATA["alerts"], ensure_ascii=False, indent=2)
    # 지원하지 않는 조회 유형이면 사용 가능한 유형을 안내합니다.
    return f"알 수 없는 조회 유형: {query_type}. 사용 가능: current, forecast, alerts"


# 전체 SmartThings 디바이스 상태를 조회하는 Tool을 정의합니다.
@tool("get_device_status")
def get_device_status() -> str:
    """전체 SmartThings 디바이스의 현재 상태를 조회합니다."""

    # 각 디바이스 상태를 담을 리스트를 초기화합니다.
    status_report = []
    # 등록된 모든 디바이스를 순회합니다.
    for device_id, device in DEVICES.items():
        # 각 디바이스의 핵심 정보를 상태 리포트에 추가합니다.
        status_report.append({
            "id": device_id, "name": device["name"],
            "room": device["room"], "type": device["type"],
            "state": device["state"]
        })
    # 전체 디바이스 상태 목록을 JSON 문자열로 반환합니다.
    return json.dumps(status_report, ensure_ascii=False, indent=2)


# 사용자 프로필과 선호 설정을 조회하는 Tool을 정의합니다.
@tool("get_user_profile")
def get_user_profile() -> str:
    """사용자 프로필과 선호 설정을 조회합니다."""
    
    # 사용자 프로필 데이터를 JSON 문자열로 반환합니다.
    return json.dumps(USER_PROFILE, ensure_ascii=False, indent=2)


# 조회용 Tool 5개가 등록되었음을 출력합니다.
print("✅ 조회 도구 5개 등록 완료!")

In [ ]:
# ─── Tool 구현: 디바이스 제어 + 씬 실행 ───

# control_device라는 이름으로 툴을 등록하고 입력 스키마로 DeviceControlInput을 사용합니다.
@tool("control_device", args_schema=DeviceControlInput)
# 개별 디바이스를 제어하는 함수를 정의합니다.
def control_device(device_id: str, command: str, value: Optional[str] = None) -> str:
    # 함수의 역할과 제어 가능한 디바이스 종류를 설명하는 문서 문자열입니다.
    """SmartThings 디바이스를 개별 제어합니다.
    조명, 에어컨, TV, 스피커, 블라인드, 도어락, 로봇청소기 등을 제어할 수 있습니다."""
    
    # 요청한 디바이스 ID가 등록된 디바이스 목록에 없는지 확인합니다.
    if device_id not in DEVICES:
        # 현재 사용 가능한 디바이스 ID 목록을 문자열로 만듭니다.
        available = ", ".join(DEVICES.keys())
        # 존재하지 않는 디바이스라는 안내 메시지를 반환합니다.
        return f"디바이스 '{device_id}' 없음. 사용 가능: {available}"

    # 제어할 디바이스 정보를 DEVICES에서 가져옵니다.
    device = DEVICES[device_id]
    # 수행된 동작 결과를 저장할 리스트를 초기화합니다.
    result_actions = []

    # 명령이 on이면 디바이스 전원을 켭니다.
    if command == "on":
        # 디바이스의 switch 상태를 on으로 변경합니다.
        device["state"]["switch"] = "on"
        # 결과 메시지에 켜짐 상태를 추가합니다.
        result_actions.append(f"{device['name']} 켜짐")
        # 디바이스 타입이 조명이라면 밝기를 100으로 설정합니다.
        if device["type"] == "light":
            device["state"]["level"] = 100

    # 명령이 off이면 디바이스 전원을 끕니다.
    elif command == "off":
        # 디바이스의 switch 상태를 off로 변경합니다.
        device["state"]["switch"] = "off"
        # 결과 메시지에 꺼짐 상태를 추가합니다.
        result_actions.append(f"{device['name']} 꺼짐")

    # 명령이 setLevel이고 값이 있으면 밝기 레벨을 설정합니다.
    elif command == "setLevel" and value:
        # 디바이스 밝기를 전달받은 값으로 설정합니다.
        device["state"]["level"] = int(value)
        # 밝기 조정 시 디바이스 전원도 켜진 상태로 설정합니다.
        device["state"]["switch"] = "on"
        # 결과 메시지에 밝기 변경 내용을 추가합니다.
        result_actions.append(f"{device['name']} 밝기 {value}%")

    # 명령이 setTemp이고 값이 있으면 온도를 설정합니다.
    elif command == "setTemp" and value:
        # 디바이스의 설정 온도를 전달받은 값으로 변경합니다.
        device["state"]["setpoint"] = int(value)
        # 온도 설정 시 디바이스 전원도 켜진 상태로 설정합니다.
        device["state"]["switch"] = "on"
        # 결과 메시지에 온도 변경 내용을 추가합니다.
        result_actions.append(f"{device['name']} 온도 {value}°C")

    # 명령이 lock이면 도어락을 잠급니다.
    elif command == "lock":
        # 디바이스의 lock 상태를 locked로 변경합니다.
        device["state"]["lock"] = "locked"
        # 결과 메시지에 잠금 상태를 추가합니다.
        result_actions.append(f"{device['name']} 잠금")

    # 명령이 unlock이면 도어락을 해제합니다.
    elif command == "unlock":
        # 디바이스의 lock 상태를 unlocked로 변경합니다.
        device["state"]["lock"] = "unlocked"
        # 결과 메시지에 잠금 해제 상태를 추가합니다.
        result_actions.append(f"{device['name']} 해제")

    # 명령이 open이면 블라인드 등을 엽니다.
    elif command == "open":
        # 디바이스의 shade 상태를 open으로 변경합니다.
        device["state"]["shade"] = "open"
        # 값이 있으면 해당 비율만큼 열고 없으면 100% 엽니다.
        device["state"]["level"] = int(value) if value else 100
        # 결과 메시지에 열림 상태를 추가합니다.
        result_actions.append(f"{device['name']} 열림 {value or 100}%")

    # 명령이 close이면 블라인드 등을 닫습니다.
    elif command == "close":
        # 디바이스의 shade 상태를 closed로 변경합니다.
        device["state"]["shade"] = "closed"
        # 닫힘 상태이므로 레벨을 0으로 설정합니다.
        device["state"]["level"] = 0
        # 결과 메시지에 닫힘 상태를 추가합니다.
        result_actions.append(f"{device['name']} 닫힘")

    # 명령이 setVolume이고 값이 있으면 볼륨을 설정합니다.
    elif command == "setVolume" and value:
        # 디바이스 볼륨을 전달받은 값으로 설정합니다.
        device["state"]["volume"] = int(value)
        # 결과 메시지에 볼륨 변경 내용을 추가합니다.
        result_actions.append(f"{device['name']} 볼륨 {value}")

    # 명령이 play이고 값이 있으면 특정 콘텐츠를 재생합니다.
    elif command == "play" and value:
        # 재생을 위해 디바이스 전원을 켭니다.
        device["state"]["switch"] = "on"
        # 현재 재생 중인 콘텐츠를 설정합니다.
        device["state"]["playing"] = value
        # 결과 메시지에 재생 내용을 추가합니다.
        result_actions.append(f"{device['name']}에서 '{value}' 재생")

    # 지원하지 않는 명령이면 오류 메시지를 반환합니다.
    else:
        # 알 수 없는 명령과 함께 입력값을 반환합니다.
        return f"알 수 없는 명령: {command} (value={value})"

    # 수행된 모든 동작 결과를 하나의 문자열로 반환합니다.
    return f"✅ 디바이스 제어 완료: {', '.join(result_actions)}"


# execute_scene라는 이름으로 툴을 등록하고 입력 스키마로 DeviceSceneInput을 사용합니다.
@tool("execute_scene", args_schema=DeviceSceneInput)
# 여러 디바이스를 한 번에 제어하는 씬 실행 함수를 정의합니다.
def execute_scene(scene_name: str) -> str:
    # 함수의 역할과 지원하는 씬 종류를 설명하는 문서 문자열입니다.
    """SmartThings 씬(Scene)을 실행하여 여러 디바이스를 한 번에 제어합니다.
    아침 브리핑, 외출 모드, 귀가 모드, 취침 모드, 릴랙스 모드, 에너지 절약 모드를 지원합니다."""

    # 씬별 설명과 실행할 액션 목록을 딕셔너리로 정의합니다.
    scenes = {
        # 아침 브리핑 씬 설정입니다.
        "morning_briefing": {
            # 씬의 사용자 표시용 설명입니다.
            "description": "🌅 아침 브리핑 모드",
            # 이 씬에서 실행할 디바이스 액션 목록입니다.
            "actions": [
                # 거실 조명을 켜고 밝기를 70으로 설정합니다.
                ("living_room_light", "on", "setLevel", "70"),
                # 거실 블라인드를 열고 개방 정도를 80으로 설정합니다.
                ("living_room_blinds", "open", "open", "80"),
                # 주방 조명을 켜고 밝기를 100으로 설정합니다.
                ("kitchen_light", "on", "setLevel", "100"),
                # 거실 스피커를 켜고 아침 재즈 플레이리스트를 재생합니다.
                ("living_room_speaker", "on", "play", "Morning Jazz Playlist"),
                # 공기청정기를 켭니다.
                ("air_purifier", "on", None, None),
                # 거실 에어컨을 켜고 사용자 선호 아침 온도로 설정합니다.
                ("living_room_ac", "on", "setTemp", str(USER_PROFILE["preferences"]["morning_temp"])),
            ]
        },
        # 외출 씬 설정입니다.
        "go_out": {
            # 씬의 사용자 표시용 설명입니다.
            "description": "🚪 외출 모드",
            # 이 씬에서 실행할 디바이스 액션 목록입니다.
            "actions": [
                # 거실 조명을 끕니다.
                ("living_room_light", "off", None, None),
                # 침실 조명을 끕니다.
                ("bedroom_light", "off", None, None),
                # 주방 조명을 끕니다.
                ("kitchen_light", "off", None, None),
                # 거실 TV를 끕니다.
                ("living_room_tv", "off", None, None),
                # 거실 스피커를 끕니다.
                ("living_room_speaker", "off", None, None),
                # 거실 에어컨을 끕니다.
                ("living_room_ac", "off", None, None),
                # 침실 에어컨을 끕니다.
                ("bedroom_ac", "off", None, None),
                # 현관문 도어락을 잠급니다.
                ("front_door_lock", "lock", None, None),
                # 거실 블라인드를 닫습니다.
                ("living_room_blinds", "close", None, None),
                # 로봇청소기를 켭니다.
                ("robot_vacuum", "on", None, None),
            ]
        },
        # 귀가 씬 설정입니다.
        "welcome_home": {
            # 씬의 사용자 표시용 설명입니다.
            "description": "🏠 귀가 모드",
            # 이 씬에서 실행할 디바이스 액션 목록입니다.
            "actions": [
                # 거실 조명을 켜고 밝기를 80으로 설정합니다.
                ("living_room_light", "on", "setLevel", "80"),
                # 거실 에어컨을 켜고 저녁 선호 온도로 설정합니다.
                ("living_room_ac", "on", "setTemp", str(USER_PROFILE["preferences"]["evening_temp"])),
                # 거실 블라인드를 닫습니다.
                ("living_room_blinds", "close", None, None),
                # 공기청정기를 켭니다.
                ("air_purifier", "on", None, None),
                # 현관문 도어락을 해제합니다.
                ("front_door_lock", "unlock", None, None),
                # 거실 스피커를 켜고 저녁용 플레이리스트를 재생합니다.
                ("living_room_speaker", "on", "play", "Lo-fi Evening Mix"),
            ]
        },
        # 취침 씬 설정입니다.
        "bedtime": {
            # 씬의 사용자 표시용 설명입니다.
            "description": "🌙 취침 모드",
            # 이 씬에서 실행할 디바이스 액션 목록입니다.
            "actions": [
                # 거실 조명을 끕니다.
                ("living_room_light", "off", None, None),
                # 주방 조명을 끕니다.
                ("kitchen_light", "off", None, None),
                # 거실 TV를 끕니다.
                ("living_room_tv", "off", None, None),
                # 거실 스피커를 끕니다.
                ("living_room_speaker", "off", None, None),
                # 침실 조명을 켜고 밝기를 10으로 설정합니다.
                ("bedroom_light", "on", "setLevel", "10"),
                # 침실 에어컨을 켜고 수면 선호 온도로 설정합니다.
                ("bedroom_ac", "on", "setTemp", str(USER_PROFILE["preferences"]["sleep_temp"])),
                # 현관문 도어락을 잠급니다.
                ("front_door_lock", "lock", None, None),
                # 거실 블라인드를 닫습니다.
                ("living_room_blinds", "close", None, None),
                # 공기청정기를 끕니다.
                ("air_purifier", "off", None, None),
                # 로봇청소기를 끕니다.
                ("robot_vacuum", "off", None, None),
            ]
        },
        # 릴랙스 씬 설정입니다.
        "relax": {
            # 씬의 사용자 표시용 설명입니다.
            "description": "☕ 릴랙스 모드",
            # 이 씬에서 실행할 디바이스 액션 목록입니다.
            "actions": [
                # 거실 조명을 켜고 밝기를 50으로 설정합니다.
                ("living_room_light", "on", "setLevel", "50"),
                # 거실 블라인드를 열고 개방 정도를 60으로 설정합니다.
                ("living_room_blinds", "open", "open", "60"),
                # 거실 에어컨을 켜고 온도를 23도로 설정합니다.
                ("living_room_ac", "on", "setTemp", "23"),
                # 거실 스피커를 켜고 주말용 플레이리스트를 재생합니다.
                ("living_room_speaker", "on", "play", "Weekend Chill Playlist"),
                # 공기청정기를 켭니다.
                ("air_purifier", "on", None, None),
            ]
        },
        # 에너지 절약 씬 설정입니다.
        "energy_saving": {
            # 씬의 사용자 표시용 설명입니다.
            "description": "⚡ 에너지 절약 모드",
            # 이 씬에서 실행할 디바이스 액션 목록입니다.
            "actions": [
                # 거실 조명을 끕니다.
                ("living_room_light", "off", None, None),
                # 침실 조명을 끕니다.
                ("bedroom_light", "off", None, None),
                # 주방 조명을 끕니다.
                ("kitchen_light", "off", None, None),
                # 거실 TV를 끕니다.
                ("living_room_tv", "off", None, None),
                # 거실 스피커를 끕니다.
                ("living_room_speaker", "off", None, None),
                # 거실 에어컨을 끕니다.
                ("living_room_ac", "off", None, None),
                # 침실 에어컨을 끕니다.
                ("bedroom_ac", "off", None, None),
                # 공기청정기를 끕니다.
                ("air_purifier", "off", None, None),
                # 현관문 도어락을 잠급니다.
                ("front_door_lock", "lock", None, None),
            ]
        },
    }

    # 입력한 씬 이름이 scenes에 정의되어 있는지 확인합니다.
    if scene_name not in scenes:
        # 사용 가능한 씬 이름 목록을 문자열로 만듭니다.
        available = ", ".join(scenes.keys())
        # 알 수 없는 씬이라는 안내 메시지를 반환합니다.
        return f"알 수 없는 씬: {scene_name}. 사용 가능: {available}"

    # 요청한 씬 정보를 가져옵니다.
    scene = scenes[scene_name]
    # 결과 리스트를 씬 설명으로 초기화합니다.
    results = [scene["description"]]

    # 씬에 정의된 각 디바이스 액션을 순회합니다.
    for device_id, switch_cmd, extra_cmd, extra_val in scene["actions"]:
        # 디바이스가 실제 등록되어 있는 경우에만 제어를 수행합니다.
        if device_id in DEVICES:
            # 현재 디바이스 정보를 가져옵니다.
            device = DEVICES[device_id]

            # 기본 on/off 명령이면 switch 상태를 변경합니다.
            if switch_cmd in ("on", "off"):
                # switch 상태를 on 또는 off로 설정합니다.
                device["state"]["switch"] = switch_cmd
            # lock 명령이면 도어락을 잠급니다.
            elif switch_cmd == "lock":
                # lock 상태를 locked로 설정합니다.
                device["state"]["lock"] = "locked"
            # unlock 명령이면 도어락을 해제합니다.
            elif switch_cmd == "unlock":
                # lock 상태를 unlocked로 설정합니다.
                device["state"]["lock"] = "unlocked"
            # open 명령이면 블라인드 상태를 엽니다.
            elif switch_cmd == "open":
                # shade 상태를 open으로 설정합니다.
                device["state"]["shade"] = "open"
            # close 명령이면 블라인드 상태를 닫습니다.
            elif switch_cmd == "close":
                # shade 상태를 closed로 설정합니다.
                device["state"]["shade"] = "closed"

            # 추가 명령이 setLevel이면 밝기를 설정합니다.
            if extra_cmd == "setLevel" and extra_val:
                # 레벨 값을 정수로 변환해 저장합니다.
                device["state"]["level"] = int(extra_val)
            # 추가 명령이 setTemp이면 온도를 설정합니다.
            elif extra_cmd == "setTemp" and extra_val:
                # 설정 온도를 정수로 변환해 저장합니다.
                device["state"]["setpoint"] = int(extra_val)
            # 추가 명령이 play이면 재생 콘텐츠를 설정합니다.
            elif extra_cmd == "play" and extra_val:
                # 현재 재생 중인 콘텐츠를 저장합니다.
                device["state"]["playing"] = extra_val
            # 추가 명령이 open이면 블라인드 개방 비율을 설정합니다.
            elif extra_cmd == "open" and extra_val:
                # 블라인드 열림 정도를 정수로 변환해 저장합니다.
                device["state"]["level"] = int(extra_val)

            # 실행 결과에 각 디바이스별 수행 내용을 추가합니다.
            results.append(f"  ✅ {device['name']}: {switch_cmd}" +
                          (f" → {extra_cmd}({extra_val})" if extra_cmd else ""))

    # 여러 줄 문자열 형태로 결과를 반환합니다.
    return "\n".join(results)


# 에이전트가 사용할 IoT 관련 도구 목록을 정의합니다.
iot_tools = [
    # 센서 조회, 디바이스 제어, 씬 실행, 일정/날씨/상태/프로필 조회 도구를 등록합니다.
    get_sensor_data, control_device, execute_scene,
    get_schedule, get_weather, get_device_status, get_user_profile,
]

# 등록된 도구 개수와 각 도구 이름을 출력합니다.
print(f"🔧 등록된 IoT 도구 ({len(iot_tools)}개): {[t.name for t in iot_tools]}")

## 6. Sub-Agent 생성 + Supervisor 에이전트 구성

### ⭐ 핵심 패턴: 서브에이전트를 `@tool`로 래핑

```python
# 1. 서브에이전트 생성
analyzer_agent = create_agent(model=llm, tools=[...], system_prompt="...")

# 2. 도구로 래핑
@tool
def analyze_context(scenario: str) -> str:
    result = analyzer_agent.invoke({"messages": [HumanMessage(content=...)]})
    return result["messages"][-1].content

# 3. Supervisor가 서브에이전트 도구 사용
supervisor = create_agent(model=llm, tools=[analyze_context], ...)
```

### 에이전트 구조

| 에이전트 | 역할 | 사용 도구 |
|---------|------|----------|
| Context Analyzer | 센서 + 환경 분석 | get_sensor_data, get_weather, get_user_profile |
| Device Controller | 디바이스 제어 | control_device, execute_scene, get_device_status |
| Briefing Agent | 맞춤 브리핑 생성 | get_weather, get_schedule, get_sensor_data, get_user_profile |
| Schedule Agent | 일정 조회/안내 | get_schedule |
| **Supervisor** | 전체 조율 | analyze_context, control_home_devices, generate_briefing, check_schedule |


In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# ─── LLM 연결: GGUF llama-server → OpenAI 호환 API (11번 참조) ───
llm = ChatOpenAI(
    model="qwen3-vl-q4",                                       # llama-server에서는 무시됨
    base_url=f"http://{SERVER_HOST}:{SERVER_PORT}/v1",          # 로컬 llama-server
    api_key="not-needed",
    temperature=0.1,
    max_tokens=4096,
)

print(f"✅ LLM 연결: http://{SERVER_HOST}:{SERVER_PORT}/v1")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 서브에이전트 1: 상황 분석 에이전트
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
context_analyzer_agent = create_agent(
    model=llm,
    tools=[get_sensor_data, get_weather, get_user_profile],
    system_prompt="""당신은 SmartThings 홈 IoT 상황 분석 전문 서브에이전트입니다.
도구를 호출할 때 인자는 반드시 짧은 키 값만 사용하세요. 긴 문장, 일정표, 임의 계획을 tool argument에 넣지 마세요.
센서 조회는 반드시 scenario_key 하나만 사용합니다: morning_commute, evening_return, bedtime, weekend_morning, away_detected.
날씨 조회는 current, forecast, alerts 중 하나만 사용합니다.

역할:
- 센서 데이터(운동센서, 위치센서, 온습도센서 등)를 분석합니다.
- 현재 날씨와 사용자 프로필을 고려하여 상황을 판단합니다.
- 분석 결과를 구조화된 형태로 반환합니다.

출력 형식:
- 현재 상황 요약
- 감지된 이벤트
- 추천 액션 (어떤 씬이 적합한지)

응답은 반드시 한국어로 작성하세요."""
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 서브에이전트 2: 디바이스 제어 에이전트
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
device_controller_agent = create_agent(
    model=llm,
    tools=[control_device, execute_scene, get_device_status],
    system_prompt="""당신은 SmartThings 홈 IoT 디바이스 제어 전문 서브에이전트입니다.
도구를 호출할 때 인자는 반드시 짧은 키 값만 사용하세요. 긴 설명문이나 여러 단계 계획을 tool argument에 넣지 마세요.
씬 실행 요청이면 execute_scene에 scene_name 하나만 전달하세요.

역할:
- 씬(Scene) 실행: 상황에 맞는 씬을 실행합니다.
- 개별 제어: 특정 디바이스를 세밀하게 제어합니다.
- 상태 확인: 현재 디바이스 상태를 조회합니다.

사용 가능한 씬:
- morning_briefing: 아침 브리핑 모드
- go_out: 외출 모드
- welcome_home: 귀가 모드
- bedtime: 취침 모드
- relax: 릴랙스 모드
- energy_saving: 에너지 절약 모드

제어 결과를 명확하게 보고하세요.
응답은 반드시 한국어로 작성하세요."""
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 서브에이전트 3: 브리핑 에이전트
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
briefing_agent = create_agent(
    model=llm,
    tools=[get_weather, get_schedule, get_sensor_data, get_user_profile],
    system_prompt="""당신은 SmartThings 홈 IoT 개인 브리핑 전문 서브에이전트입니다.
도구를 호출할 때 인자는 반드시 짧은 키 값만 사용하세요. 존재하지 않는 일정이나 반복 시간표를 만들지 마세요.
일정 조회는 morning, afternoon, evening, all_day, next 중 하나만 사용합니다.
날씨 조회는 current, forecast, alerts 중 하나만 사용합니다.

역할:
- 날씨, 일정, 센서 데이터를 종합하여 사용자에게 맞춤 브리핑을 제공합니다.
- 브리핑은 간결하고 실용적이어야 합니다.
- 중요한 일정은 강조하고, 날씨에 따른 조언을 포함합니다.

브리핑 구성:
1. 인사 (시간대별)
2. 날씨 요약 + 미세먼지
3. 오늘의 주요 일정
4. 특별 알림 (기상특보, 중요 메모 등)
5. 맞춤 조언

응답은 반드시 한국어로 작성하세요. 친근하면서도 전문적인 톤으로 작성하세요."""
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 서브에이전트 4: 일정 관리 에이전트
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
schedule_agent = create_agent(
    model=llm,
    tools=[get_schedule],
    system_prompt="""당신은 사용자의 일정 확인과 시간 관리 안내를 담당하는 서브에이전트입니다.
도구를 호출할 때 time_range는 morning, afternoon, evening, all_day, next 중 하나만 사용하세요.
도구 결과에 없는 일정을 만들거나 반복 시간표를 생성하지 마세요.

역할:
- 사용자의 일정을 시간대별로 안내합니다.
- 다음 일정까지의 남은 시간을 계산합니다.
- 일정 간 충돌이나 촉박한 일정을 알려줍니다.
- 이동 시간이 필요한 일정의 경우 출발 시간을 조언합니다.

응답은 반드시 한국어로 작성하세요."""
)

print("✅ 서브에이전트 4개 생성 완료!")
print("   1️⃣ Context Analyzer (상황 분석)")
print("   2️⃣ Device Controller (디바이스 제어)")
print("   3️⃣ Briefing Agent (브리핑)")
print("   4️⃣ Schedule Agent (일정)")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 서브에이전트를 도구(Tool)로 래핑
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@tool
def analyze_context(sensor_scenario: str) -> str:
    """센서 데이터와 환경 정보를 분석하여 현재 사용자 상황을 파악합니다.
    scenario: morning_commute, evening_return, bedtime, weekend_morning, away_detected"""
    result = context_analyzer_agent.invoke({
        "messages": [HumanMessage(content=(
            f"scenario_key={sensor_scenario}. get_sensor_data에는 이 키만 전달하세요. 날씨는 current만 조회하세요. 도구 결과만 근거로 3줄 이내로 분석하세요."
        ))]
    })
    return result["messages"][-1].content


@tool
def control_home_devices(command_description: str) -> str:
    """SmartThings 디바이스를 제어합니다. 씬 실행 또는 개별 디바이스 제어가 가능합니다.
    예: '귀가 모드 실행', '거실 조명 50%로 조절', '에어컨 22도로 설정'"""
    result = device_controller_agent.invoke({
        "messages": [HumanMessage(content=(
            f"요청: {command_description}. 씬 요청이면 execute_scene에 정확한 scene_name 하나만 전달하세요. 긴 문장을 tool argument로 쓰지 마세요."
        ))]
    })
    return result["messages"][-1].content


@tool
def generate_briefing(context_info: str) -> str:
    """현재 상황에 맞는 개인 맞춤 브리핑을 생성합니다.
    날씨, 일정, 센서 상태를 종합하여 브리핑합니다."""
    result = briefing_agent.invoke({
        "messages": [HumanMessage(content=(
            f"다음 상황 정보를 바탕으로 날씨, 일정, 센서 상태를 종합한 맞춤 브리핑을 작성하세요:\n{context_info}"
        ))]
    })
    return result["messages"][-1].content


@tool
def check_schedule(time_range: str) -> str:
    """일정을 조회하고 안내합니다.
    time_range: morning(오전), afternoon(오후), evening(저녁), all_day(전체), next(다음 일정)"""
    result = schedule_agent.invoke({
        "messages": [HumanMessage(content=(
            f"time_range={time_range}. get_schedule에는 이 값 하나만 전달하세요. 도구 결과에 없는 일정은 만들지 마세요."
        ))]
    })
    return result["messages"][-1].content


print("✅ 서브에이전트 → Tool 래핑 완료!")
print("   🔧 analyze_context → Context Analyzer 호출")
print("   🔧 control_home_devices → Device Controller 호출")
print("   🔧 generate_briefing → Briefing Agent 호출")
print("   🔧 check_schedule → Schedule Agent 호출")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Supervisor 에이전트 (메인)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

supervisor_tools = [analyze_context, control_home_devices, generate_briefing, check_schedule]

supervisor = create_agent(
    model=llm,
    tools=supervisor_tools,
    system_prompt="""당신은 SmartThings 기반 홈 IoT 멀티 디바이스 제어 Supervisor Agent입니다.

## Tool Calling 규칙
- tool argument에는 반드시 짧은 값만 넣으세요.
- analyze_context에는 sensor_scenario 키 하나만 전달하세요. 예: morning_commute
- control_home_devices에는 실행할 씬 이름 중심의 짧은 요청만 전달하세요. 예: go_out 씬 실행
- check_schedule에는 morning, afternoon, evening, all_day, next 중 하나만 전달하세요.
- generate_briefing에는 앞선 도구 결과 요약만 전달하세요.
- tool argument에 긴 한국어 문단, 반복 시간표, 임의 일정, 여러 단계 계획을 절대 넣지 마세요.
- 도구 결과에 없는 사실을 만들지 마세요.

## 역할
사용자의 요청과 센서 시나리오를 해석해 상황 분석, 디바이스 제어, 브리핑, 일정 확인 서브에이전트를 적절한 순서로 호출합니다.

## 사용 가능한 서브에이전트 (도구)
1. **analyze_context**: 센서 데이터 분석 → 상황 파악
2. **control_home_devices**: 디바이스 제어 (씬 실행 또는 개별 제어)
3. **generate_briefing**: 상황별 맞춤 브리핑 생성
4. **check_schedule**: 일정 조회 및 안내

## 처리 흐름
1. 먼저 analyze_context로 상황을 분석합니다.
2. 분석 결과에 따라 적절한 control_home_devices로 디바이스를 제어합니다.
3. generate_briefing 또는 check_schedule로 사용자에게 정보를 제공합니다.
4. 최종 결과를 종합하여 사용자에게 보고합니다.

## 시나리오별 처리 가이드
- 아침 출근: analyze_context → control_home_devices(go_out 씬) → generate_briefing(아침 브리핑)
- 저녁 퇴근: analyze_context → control_home_devices(welcome_home 씬) → check_schedule(evening) → generate_briefing
- 취침: analyze_context → control_home_devices(bedtime 씬)
- 주말 아침: analyze_context → control_home_devices(relax 씬) → generate_briefing
- 외출 감지: analyze_context → control_home_devices(energy_saving 씬)

## 최종 출력
- 수행한 상황 분석, 실행한 디바이스 제어, 확인한 일정/브리핑을 구분해 요약하세요.
- 실제 도구 결과에 근거해서 작성하고, 확인되지 않은 내용은 추측하지 마세요.
- 사용자가 바로 이해할 수 있도록 한국어로 간결하게 보고하세요."""
)

print("✅ Supervisor 에이전트 생성 완료!")
print(f"   🔧 서브에이전트 도구: {[t.name for t in supervisor_tools]}")


## 7. 에이전트 실행 함수

시나리오별 실행 + 메시지 추적 + 디바이스 상태 시각화를 제공합니다.


In [ ]:
def run_scenario(scenario_key: str, verbose: bool = True):
    """시나리오를 실행하고 결과를 추적합니다."""
    if scenario_key not in SENSOR_SCENARIOS:
        print(f"❌ 알 수 없는 시나리오: {scenario_key}")
        print(f"   사용 가능: {', '.join(SENSOR_SCENARIOS.keys())}")
        return

    scenario = SENSOR_SCENARIOS[scenario_key]

    print("\n" + "=" * 70)
    print(f"🏠 SmartThings 홈 IoT 시나리오 실행")
    print("=" * 70)
    print(f"📋 시나리오: {scenario['description']}")
    print(f"⏰ 시간: {scenario['timestamp']}")
    print(f"🔑 키: {scenario_key}")
    print("=" * 70)

    # 시나리오에 맞는 요청 구성
    scenario_prompts = {
        "morning_commute": (
            f"현재 시각은 {scenario['timestamp']}이고, "
            f"운동 센서가 현관에서 활동을 감지했으며, 위치 센서가 사용자가 출근 중임을 나타냅니다. "
            f"시나리오 키는 'morning_commute'입니다. "
            f"1) 상황을 분석하고, 2) 외출 모드(go_out)를 실행하고, "
            f"3) 아침 브리핑을 생성해주세요."
        ),
        "evening_return": (
            f"현재 시각은 {scenario['timestamp']}이고, "
            f"위치 센서가 사용자가 퇴근 중이며 집에서 3.2km 거리에 있음을 감지했습니다. "
            f"시나리오 키는 'evening_return'입니다. "
            f"1) 상황을 분석하고, 2) 귀가 모드(welcome_home)를 실행하고, "
            f"3) 저녁 일정을 확인하고, 4) 저녁 브리핑을 생성해주세요."
        ),
        "bedtime": (
            f"현재 시각은 {scenario['timestamp']}이고, "
            f"수면 센서가 수면 준비 상태를 감지했고, 침실에서 활동이 감지되었습니다. "
            f"시나리오 키는 'bedtime'입니다. "
            f"1) 상황을 분석하고, 2) 취침 모드(bedtime)를 실행해주세요."
        ),
        "weekend_morning": (
            f"현재 시각은 {scenario['timestamp']}이고, 오늘은 주말(토요일)입니다. "
            f"거실에서 활동이 감지되었습니다. "
            f"시나리오 키는 'weekend_morning'입니다. "
            f"1) 상황을 분석하고, 2) 릴랙스 모드(relax)를 실행하고, "
            f"3) 주말 브리핑을 생성해주세요."
        ),
        "away_detected": (
            f"현재 시각은 {scenario['timestamp']}이고, "
            f"30분간 집 안에서 활동이 감지되지 않았으며, 위치 센서가 사용자의 외출을 확인했습니다. "
            f"시나리오 키는 'away_detected'입니다. "
            f"1) 상황을 분석하고, 2) 에너지 절약 모드(energy_saving)를 실행해주세요."
        ),
    }

    request = scenario_prompts[scenario_key]

    # 에이전트 실행
    result = supervisor.invoke({"messages": [HumanMessage(content=request)]})

    # 실행 과정 추적
    if verbose:
        print("\n" + "-" * 70)
        print("🔍 실행 과정 (메시지 추적)")
        print("-" * 70)

        for i, msg in enumerate(result["messages"]):
            if isinstance(msg, HumanMessage):
                content = msg.content if isinstance(msg.content, str) else str(msg.content)
                print(f"\n[{i}] 👤 User: {content[:80]}...")
            elif isinstance(msg, AIMessage):
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    for tc in msg.tool_calls:
                        args_str = json.dumps(tc['args'], ensure_ascii=False)
                        print(f"\n[{i}] 🔧 Tool Call: {tc['name']}")
                        print(f"    └─ Args: {args_str[:120]}...")
                elif msg.content:
                    content = msg.content[:150] + "..." if len(msg.content) > 150 else msg.content
                    print(f"\n[{i}] 🤖 AI: {content}")
            elif isinstance(msg, ToolMessage):
                content = msg.content[:120] + "..." if len(msg.content) > 120 else msg.content
                print(f"\n[{i}] 📤 Tool Result [{msg.name}]: {content}")

    # 최종 결과
    print("\n" + "=" * 70)
    print("✅ 최종 결과")
    print("=" * 70)
    final = result["messages"][-1].content
    print(final)

    # 디바이스 상태 변화 요약
    print("\n" + "-" * 70)
    print("📊 디바이스 상태 (씬 실행 후)")
    print("-" * 70)
    for device_id, device in DEVICES.items():
        switch_state = device["state"].get("switch", "-")
        extra_info = ""
        if device["type"] == "light" and switch_state == "on":
            extra_info = f" (밝기: {device['state'].get('level', '-')}%)"
        elif device["type"] == "airConditioner" and switch_state == "on":
            extra_info = f" (설정: {device['state'].get('setpoint', '-')}°C)"
        elif device["type"] == "lock":
            extra_info = f" ({device['state'].get('lock', '-')})"
            switch_state = "-"
        elif device["type"] == "windowShade":
            extra_info = f" ({device['state'].get('shade', '-')})"
            switch_state = "-"
        elif device["type"] == "speaker" and switch_state == "on":
            playing = device["state"].get("playing", "-")
            extra_info = f" (재생: {playing})"

        icon = "🟢" if switch_state == "on" else "🔴" if switch_state == "off" else "⚪"
        print(f"  {icon} {device['name']:12s} [{device['room']:4s}]: {switch_state}{extra_info}")

    return result


def _reset_devices():
    """디바이스 상태를 초기값으로 리셋합니다."""
    for device_id, device in DEVICES.items():
        if "switch" in device["state"]:
            device["state"]["switch"] = "off"
        if "level" in device["state"]:
            device["state"]["level"] = 0
        if "lock" in device["state"]:
            device["state"]["lock"] = "locked"
        if "shade" in device["state"]:
            device["state"]["shade"] = "closed"
            device["state"]["level"] = 0
        if "playing" in device["state"]:
            device["state"]["playing"] = None


def chat(message: str):
    """사용자가 자유롭게 홈 IoT 에이전트와 대화합니다."""
    print(f"\n{'='*60}")
    print(f"💬 사용자: {message}")
    print(f"{'='*60}")
    result = supervisor.invoke({"messages": [HumanMessage(content=message)]})
    print(f"\n🤖 AI 비서:\n{result['messages'][-1].content}")
    return result


print("✅ 실행 함수 정의 완료!")
print()
print("📌 사용법:")
print('  run_scenario("morning_commute")   # 아침 출근 시나리오')
print('  run_scenario("evening_return")    # 저녁 퇴근 시나리오')
print('  run_scenario("bedtime")           # 취침 시나리오')
print('  run_scenario("weekend_morning")   # 주말 아침 시나리오')
print('  run_scenario("away_detected")     # 외출 감지 시나리오')


## 9. 시나리오 실행 — ① 아침 출근

📌 **아침 9시, 운동 센서 감지, 출근 중** → 하루 브리핑 + 외출 모드


In [ ]:
run_scenario("morning_commute")

## 10. 시나리오 실행 — ② 저녁 퇴근

📌 **저녁 8시, 위치 센서 감지, 퇴근 중** → 일정 안내 + 귀가 모드


In [ ]:
_reset_devices()
run_scenario("evening_return")

## 11. 시나리오 실행 — ③ 취침

📌 **밤 11시, 수면 센서 감지, 침실 활동** → 취침 모드 전환


In [ ]:
_reset_devices()
run_scenario("bedtime")

## 12. 시나리오 실행 — ④ 주말 아침

📌 **주말 오전 10시, 거실 활동 감지** → 릴랙스 모드


In [ ]:
_reset_devices()
run_scenario("weekend_morning")

## 13. 시나리오 실행 — ⑤ 외출 감지

📌 **오후 2시, 30분간 활동 없음 + 외출 감지** → 에너지 절약 모드


In [ ]:
_reset_devices()
run_scenario("away_detected")

## 14. 자유 대화 모드

사용자가 직접 홈 IoT 에이전트에게 명령할 수 있습니다.


In [ ]:
# 자유 대화 예시 (주석 해제하여 실행)
chat("거실 조명을 50%로 줄여줘")
chat("내일 일정이 어떻게 돼?")
chat("에어컨 온도를 23도로 맞춰줘")
chat("지금 집 안 상태가 어때?")


## 15. (선택) 서버 종료

작업이 끝나면 llama-server를 종료합니다.


In [ ]:
# 서버 종료
server_process.terminate()
server_process.wait()
print("🛑 llama-server 종료 완료")
